In [ ]:
import torch
import torch.optim as optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os
import numpy as np
import torchvision.utils as vutils
import random
import torch.optim as optim
import torch.nn.functional as F
import torch.nn as nn

In [ ]:
# Fix the directory paths
noisy_dir = "data/crops/noisy"
clean_dir = "data/crops/clean"

In [ ]:
IMG_HEIGHT = 512
IMG_WIDTH = 512
BATCH_SIZE = 16
EPOCHS = 10
LEARNING_RATE = 1e-4

In [ ]:
class TinyDenoisingModel(nn.Module):
    def __init__(self):
        super(DenoisingModel, self).__init__()
        
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1),  # Changed input channels to 3 for RGB
            nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(128),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout(0.5),
        )

        self.decoder = nn.Sequential(
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 3, kernel_size=3, padding=1),  # Output is 3 channels for RGB
            nn.Sigmoid(),
            nn.Upsample(scale_factor=2),
        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x

In [ ]:
class UNetDenoisingModel(nn.Module):
    def __init__(self):
        super(UNetDenoisingModel, self).__init__()
        
        # Encoder
        self.enc1 = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(64)
        )
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.enc2 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(128)
        )
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.enc3 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(256)
        )
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Bridge
        self.bridge = nn.Sequential(
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(512)
        )
        
        # Decoder
        self.upconv3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec3 = nn.Sequential(
            nn.Conv2d(512, 256, kernel_size=3, padding=1),  # 512 because of skip connection
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(256)
        )
        
        self.upconv2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec2 = nn.Sequential(
            nn.Conv2d(256, 128, kernel_size=3, padding=1),  # 256 because of skip connection
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(128)
        )
        
        self.upconv1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec1 = nn.Sequential(
            nn.Conv2d(128, 64, kernel_size=3, padding=1),  # 128 because of skip connection
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(64)
        )
        
        # Final output
        self.final = nn.Sequential(
            nn.Conv2d(64, 3, kernel_size=1),
            nn.Sigmoid()
        )
        
    def forward(self, x):
        # Encoder
        enc1 = self.enc1(x)
        pool1 = self.pool1(enc1)
        
        enc2 = self.enc2(pool1)
        pool2 = self.pool2(enc2)
        
        enc3 = self.enc3(pool2)
        pool3 = self.pool3(enc3)
        
        # Bridge
        bridge = self.bridge(pool3)
        
        # Decoder with skip connections
        up3 = self.upconv3(bridge)
        up3 = torch.cat([up3, enc3], dim=1)  # Skip connection
        dec3 = self.dec3(up3)
        
        up2 = self.upconv2(dec3)
        up2 = torch.cat([up2, enc2], dim=1)  # Skip connection
        dec2 = self.dec2(up2)
        
        up1 = self.upconv1(dec2)
        up1 = torch.cat([up1, enc1], dim=1)  # Skip connection
        dec1 = self.dec1(up1)
        
        return self.final(dec1)

In [ ]:
class DocumentDenoisingDataset(Dataset):
    def __init__(self, noisy_dir, clean_dir, transform=None):
        self.noisy_images = sorted(os.listdir(noisy_dir))
        self.clean_images = sorted(os.listdir(clean_dir))
        self.noisy_dir = noisy_dir
        self.clean_dir = clean_dir
        self.transform = transform

    def __len__(self):
        return len(self.noisy_images)

    def __getitem__(self, idx):
        noisy_image_path = os.path.join(self.noisy_dir, self.noisy_images[idx])
        clean_image_path = os.path.join(self.clean_dir, self.clean_images[idx])
        
        # Load images in RGB mode
        noisy = Image.open(noisy_image_path).convert("RGB")
        clean = Image.open(clean_image_path).convert("RGB")
        
        # Apply transforms (resize, to tensor)
        if self.transform:
            noisy = self.transform(noisy)
            clean = self.transform(clean)

        return noisy, clean

In [ ]:
# Transforms to apply to each image
transform = transforms.Compose([
    transforms.Resize((256, 256)),  # Resize to (256, 256)
    transforms.ToTensor(),  # Convert to tensor (C, H, W)
])

In [ ]:
# Update dataset creation
dataset = DocumentDenoisingDataset(noisy_dir, clean_dir, transform)  # Use correct directory variables
dataloader = torch.utils.data.DataLoader(dataset, batch_size=16, shuffle=True)

In [ ]:
def visualize_random_input_output(inputs, outputs, epoch):
    idx = random.randint(0, inputs.size(0) - 1)  # Select a random index from the batch
    input_image = inputs[idx].detach().cpu()  # Shape: (3, H, W)
    output_image = outputs[idx].detach().cpu()  # Shape: (3, H, W)
    
    # Rearrange from (C, H, W) to (H, W, C) for display
    input_image = input_image.permute(1, 2, 0)
    output_image = output_image.permute(1, 2, 0)

    plt.figure(figsize=(8, 4))

    plt.subplot(1, 2, 1)
    plt.title(f"Epoch {epoch+1} - Noisy Input")
    plt.imshow(input_image)  # Removed cmap="gray"
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.title(f"Epoch {epoch+1} - Model Output")
    plt.imshow(output_image)  # Removed cmap="gray"
    plt.axis('off')

    plt.show()

In [ ]:
# Training loop with loss tracking
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = UNetDenoisingModel()
model.to(device)

# Loss and optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

train_losses = []  # Add this to store losses

print(device)

for epoch in range(50):
    print(f"Epoch [{epoch+1}/50]")
    model.train()
    running_loss = 0.0

    for batch_idx, (noisy, clean) in enumerate(dataloader):
        noisy, clean = noisy.to(device), clean.to(device)
        
        optimizer.zero_grad()
        outputs = model(noisy)
        loss = criterion(outputs, clean)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()

    epoch_loss = running_loss/len(dataloader)
    train_losses.append(epoch_loss)  # Store the average loss for this epoch
    
    # Visualize one random input-output pair
    visualize_random_input_output(noisy, outputs, epoch)
    print(f"Epoch [{epoch+1}/10], Loss: {epoch_loss:.4f}")

# Save the trained model
torch.save(model.state_dict(), "denoising_model.pth")

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(train_losses) + 1), train_losses, marker='o')
plt.title('Training Loss Over Time')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True)
plt.show()

In [ ]:
def denoise_large_image(image_path, patch_size=(IMG_HEIGHT, IMG_WIDTH), overlap=32):
    """
    Denoise a large image by processing it in patches and reconstructing.
    
    Args:
        image_path: Path to the image file
        patch_size: Tuple of (height, width) for patches
        overlap: Number of pixels to overlap between patches
    """
    # Load image and convert to RGB
    img = Image.open(image_path).convert('RGB')
    img_array = np.array(img)
    
    # Get dimensions
    height, width, channels = img_array.shape
    
    # Initialize output array
    output = np.zeros_like(img_array, dtype=np.float32)
    weight = np.zeros((height, width), dtype=np.float32)
    
    # Prepare model
    model.eval()
    
    # Calculate steps for overlapping patches
    h_stride = patch_size[0] - overlap
    w_stride = patch_size[1] - overlap
    
    for h in range(0, height - overlap, h_stride):
        for w in range(0, width - overlap, w_stride):
            # Calculate patch boundaries
            h_start = h
            w_start = w
            h_end = min(h + patch_size[0], height)
            w_end = min(w + patch_size[1], width)
            
            # Extract patch
            patch = img_array[h_start:h_end, w_start:w_end, :]
            
            # Create a tensor of the correct size
            patch_tensor = torch.zeros((1, 3, patch_size[0], patch_size[1]), device=device)
            
            # Place the actual patch data into the tensor
            for c in range(channels):
                patch_tensor[0, c, :patch.shape[0], :patch.shape[1]] = torch.from_numpy(patch[:, :, c])
            
            # Normalize to [0, 1]
            patch_tensor = patch_tensor / 255.0
            
            with torch.no_grad():
                denoised_patch = model(patch_tensor)
            
            # Convert back to numpy and denormalize
            denoised_patch = (denoised_patch.squeeze().cpu().numpy() * 255.0)
            
            # Crop back to original patch size
            denoised_patch = denoised_patch[:, :h_end-h_start, :w_end-w_start]
            
            # Add to output using weight matrix for overlapping regions
            output[h_start:h_end, w_start:w_end, :] += denoised_patch.transpose(1, 2, 0)
            weight[h_start:h_end, w_start:w_end] += 1
    
    # Average overlapping regions
    for c in range(channels):
        output[:, :, c] = np.divide(output[:, :, c], weight, where=weight != 0)
    
    # Clip values to valid range
    output = np.clip(output, 0, 255)

    # Plot results
    plt.figure(figsize=(20, 10))
    
    plt.subplot(1, 2, 1)
    plt.title("Original Image")
    plt.imshow(img_array)
    plt.axis('off')
    
    plt.subplot(1, 2, 2)
    plt.title("Denoised Image")
    plt.imshow(output.astype(np.uint8))
    plt.axis('off')
    
    plt.show()
    
    return output

# Example usage:
image_path = "data/source_pdfs/sample_test_image.jpg"  # Replace with your image path
denoised_image = denoise_large_image(image_path, patch_size=(IMG_HEIGHT, IMG_WIDTH), overlap=32)

# Optionally save the result
denoised_pil = Image.fromarray(denoised_image.astype(np.uint8))
denoised_pil.save("denoised_large_image.png")